Entrenar Modelo en Google Colab

In [4]:
# Instalar kaggle y librerías necesarias
!pip install -q kaggle pandas

# Subir kaggle.json
from google.colab import files
files.upload()  # <- subí el archivo kaggle.json cuando lo pida

# Mover a la ubicación correcta
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


Saving kaggle.json to kaggle.json


In [10]:
import os
import shutil
import pandas as pd
from kaggle.api.kaggle_api_extended import KaggleApi

# Descargar dataset
api = KaggleApi()
api.authenticate()

DATASET_NAME = "archanghosh/yugioh-database"
DOWNLOAD_DIR = "raw_yugioh"
OUTPUT_DIR = "yu_dataset_cleaned"

api.dataset_download_files(DATASET_NAME, path=DOWNLOAD_DIR, unzip=True)

# Leer y preparar carpetas
df = pd.read_csv(f"{DOWNLOAD_DIR}/Yugi_db_cleaned.csv")
os.makedirs(OUTPUT_DIR, exist_ok=True)

for label in ["Monster", "Spell", "Trap"]:
    os.makedirs(f"{OUTPUT_DIR}/{label}", exist_ok=True)

image_dir = os.path.join(DOWNLOAD_DIR, "Yugi_images")

for _, row in df.iterrows():
    card_type = row.get("Card type", "")
    image_name = row.get("Image_name", "")

    if not isinstance(card_type, str) or not isinstance(image_name, str):
        continue

    if "Monster" in card_type:
        label = "Monster"
    elif "Spell" in card_type:
        label = "Spell"
    elif "Trap" in card_type:
        label = "Trap"
    else:
        continue

    src = os.path.join(image_dir, image_name)
    dst = os.path.join(OUTPUT_DIR, label, image_name)

    if os.path.exists(src):
        shutil.copy2(src, dst)


Dataset URL: https://www.kaggle.com/datasets/archanghosh/yugioh-database


**Crear y entrenar al modelo**

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models

# Parámetros
OUTPUT_DIR = 'yu_dataset_cleaned'
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

# Generadores
train_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_generator = train_datagen.flow_from_directory(
    OUTPUT_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

val_generator = train_datagen.flow_from_directory(
    OUTPUT_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

# Modelo CNN básico
model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(128, 128, 3)), # capa convolutional
    layers.MaxPooling2D(2,2), # pooling layer
    layers.Conv2D(64, (3,3), activation='relu'), # capa convolutional
    layers.MaxPooling2D(2,2), # poolen layer, siempre despues de una capa convlutional
    layers.Flatten(), # se aplanan las capas a un único vector
    layers.Dense(128, activation='relu'), # fully connected layer, se puede tener más después
    layers.Dense(3, activation='softmax')  # 3 clases, se aplica una función softmax para obtener las probabilidades de estas
])

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Entrenar
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10
)

# 1. Guardar en formato SavedModel, para evitar errores de compatibilidad con otras versiones de tensorflow
saved_model_dir = "yugioh_model"
model.save(saved_model_dir)  # Esto crea una carpeta

# 2. Comprimir la carpeta en un zip
shutil.make_archive(saved_model_dir, 'zip', saved_model_dir)

# 3. Mostrar enlace de descarga
from google.colab import files
files.download(f"{saved_model_dir}.zip")

Found 6700 images belonging to 3 classes.
Found 1673 images belonging to 3 classes.


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
210/210 ━━━━━━━━━━━━━━━━━━━━ 280s 1s/step - accuracy: 0.9004 - loss: 0.5475 - val_accuracy: 0.9988 - val_loss: 0.0043
Epoch 2/10
210/210 ━━━━━━━━━━━━━━━━━━━━ 325s 1s/step - accuracy: 0.9958 - loss: 0.0138 - val_accuracy: 1.0000 - val_loss: 3.9983e-04
Epoch 3/10
210/210 ━━━━━━━━━━━━━━━━━━━━ 261s 1s/step - accuracy: 1.0000 - loss: 1.4305e-04 - val_accuracy: 1.0000 - val_loss: 7.9411e-05
Epoch 4/10
210/210 ━━━━━━━━━━━━━━━━━━━━ 267s 1s/step - accuracy: 1.0000 - loss: 6.5608e-05 - val_accuracy: 1.0000 - val_loss: 1.6424e-04
Epoch 5/10
210/210 ━━━━━━━━━━━━━━━━━━━━ 267s 1s/step - accuracy: 1.0000 - loss: 1.7008e-05 - val_accuracy: 1.0000 - val_loss: 1.7382e-04
Epoch 6/10
210/210 ━━━━━━━━━━━━━━━━━━━━ 274s 1s/step - accuracy: 1.0000 - loss: 1.2502e-05 - val_accuracy: 1.0000 - val_loss: 2.5170e-04
Epoch 7/10
210/210 ━━━━━━━━━━━━━━━━━━━━ 273s 1s/step - accuracy: 1.0000 - loss: 1.0520e-05 - val_accuracy: 1.0000 - val_loss: 3.0915e-04
Epoch 8/10
210/210 ━━━━━━━━━━━━━━━━━━━━ 271s 1s/step 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>